# CNN no Dataset CIFAR-10

Comparação entre a **LightCNN** (customizada) e a **ResNet18** (pré-treinada) no dataset CIFAR-10.

- **CIFAR-10**: 50.000 imagens de treino e 10.000 de teste, 32x32 pixels, coloridas (RGB), 10 classes.
- Classes: airplane, automobile, bird, cat, deer, dog, frog, horse, ship, truck.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms
from torchvision.models import resnet18, ResNet18_Weights

from models import LightCNN
from utils import (
    get_device, train_model, plot_training_curves,
    plot_comparison_bar, get_predictions, plot_confusion_matrix,
    visualize_activations, print_summary_table,
)

device = get_device()
print(f"Dispositivo: {device}")

## 1. Carregamento e Preparação dos Dados

In [ ]:
BATCH_SIZE = 64
NUM_WORKERS = 0  # Windows-safe

CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD = (0.2470, 0.2435, 0.2616)

transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD),
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD),
])

train_dataset = torchvision.datasets.CIFAR10(
    root='./data', train=True, download=True, transform=transform_train
)
test_dataset = torchvision.datasets.CIFAR10(
    root='./data', train=False, download=True, transform=transform_test
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=True)

CLASSES = ['airplane', 'automobile', 'bird', 'cat', 'deer',
           'dog', 'frog', 'horse', 'ship', 'truck']

print(f"Treino: {len(train_dataset)} amostras | Teste: {len(test_dataset)} amostras")

## 2. Visualização de Amostras

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Carregar sem normalização para visualização
viz_dataset = torchvision.datasets.CIFAR10(
    root='./data', train=True, download=False,
    transform=transforms.ToTensor()
)

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    img, label = viz_dataset[i]
    ax.imshow(img.permute(1, 2, 0).numpy())
    ax.set_title(f'{CLASSES[label]}')
    ax.axis('off')
plt.suptitle('Amostras do CIFAR-10', fontsize=14)
plt.tight_layout()
plt.show()

## 3. Treinamento da LightCNN

In [ ]:
EPOCHS_CUSTOM = 30
LR = 1e-3

model_custom = LightCNN(in_channels=3, num_classes=10, img_size=32).to(device)
criterion = nn.CrossEntropyLoss()
optimizer_custom = optim.Adam(model_custom.parameters(), lr=LR)
scheduler_custom = optim.lr_scheduler.StepLR(optimizer_custom, step_size=15, gamma=0.1)

print(f"Parâmetros da LightCNN: {sum(p.numel() for p in model_custom.parameters()):,}")

history_custom = train_model(
    model_custom, train_loader, test_loader, criterion, optimizer_custom,
    device, epochs=EPOCHS_CUSTOM, scheduler=scheduler_custom,
    model_name="LightCNN"
)

In [ ]:
plot_training_curves(history_custom, title="LightCNN — CIFAR-10")

## 4. Fine-tuning da ResNet18

In [ ]:
EPOCHS_RESNET = 15

model_resnet = resnet18(weights=ResNet18_Weights.DEFAULT)

# Congelar todas as camadas exceto layer4 e fc
for name, param in model_resnet.named_parameters():
    if "layer4" not in name and "fc" not in name:
        param.requires_grad = False

# Adaptar classificador para 10 classes
model_resnet.fc = nn.Linear(model_resnet.fc.in_features, 10)
model_resnet = model_resnet.to(device)

trainable = sum(p.numel() for p in model_resnet.parameters() if p.requires_grad)
total = sum(p.numel() for p in model_resnet.parameters())
print(f"ResNet18 — Total: {total:,} | Treináveis: {trainable:,}")

optimizer_resnet = optim.Adam(
    filter(lambda p: p.requires_grad, model_resnet.parameters()), lr=1e-3
)
scheduler_resnet = optim.lr_scheduler.StepLR(optimizer_resnet, step_size=8, gamma=0.1)

history_resnet = train_model(
    model_resnet, train_loader, test_loader, criterion, optimizer_resnet,
    device, epochs=EPOCHS_RESNET, scheduler=scheduler_resnet,
    model_name="ResNet18"
)

In [ ]:
plot_training_curves(history_resnet, title="ResNet18 — CIFAR-10")

## 5. Comparação de Resultados

In [ ]:
results = {"LightCNN": history_custom, "ResNet18": history_resnet}

print_summary_table(results)
plot_comparison_bar(results, title="Comparação de Acurácia — CIFAR-10")

## 6. Matrizes de Confusão

In [ ]:
y_true_c, y_pred_c = get_predictions(model_custom, test_loader, device)
plot_confusion_matrix(y_true_c, y_pred_c, CLASSES,
                      title="Matriz de Confusão — LightCNN (CIFAR-10)")

y_true_r, y_pred_r = get_predictions(model_resnet, test_loader, device)
plot_confusion_matrix(y_true_r, y_pred_r, CLASSES,
                      title="Matriz de Confusão — ResNet18 (CIFAR-10)")

## 7. Visualização das Ativações dos Kernels (DESAFIO)

In [ ]:
# Selecionar uma imagem de exemplo (sem augmentation)
sample_img, sample_label = test_dataset[0]
sample_img_batch = sample_img.unsqueeze(0)  # (1, 3, 32, 32)

# Visualizar a imagem original (desnormalizar)
img_show = sample_img.clone()
for c in range(3):
    img_show[c] = img_show[c] * CIFAR10_STD[c] + CIFAR10_MEAN[c]
img_show = img_show.clamp(0, 1)

plt.figure(figsize=(2, 2))
plt.imshow(img_show.permute(1, 2, 0).numpy())
plt.title(f'Imagem de entrada — {CLASSES[sample_label]}')
plt.axis('off')
plt.show()

print("\nAtivações da LightCNN:")
visualize_activations(model_custom, sample_img_batch, device,
                      title_prefix="LightCNN CIFAR-10 — ")

## 8. Curvas de Treino Comparativas

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Loss
ax1.plot(history_custom['val_loss'], label='LightCNN', marker='o', markersize=3)
ax1.plot(history_resnet['val_loss'], label='ResNet18', marker='s', markersize=3)
ax1.set_xlabel('Época')
ax1.set_ylabel('Loss (Validação)')
ax1.set_title('Loss de Validação — CIFAR-10')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Acurácia
ax2.plot([a * 100 for a in history_custom['val_acc']], label='LightCNN', marker='o', markersize=3)
ax2.plot([a * 100 for a in history_resnet['val_acc']], label='ResNet18', marker='s', markersize=3)
ax2.set_xlabel('Época')
ax2.set_ylabel('Acurácia (%) (Validação)')
ax2.set_title('Acurácia de Validação — CIFAR-10')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 9. Análise e Discussão

### Resultados Observados

No CIFAR-10, a diferença de desempenho entre a LightCNN e a ResNet18 é significativamente mais acentuada do que no MNIST. A ResNet18 pré-treinada alcança acurácia superior graças ao *transfer learning*.

### Por que a ResNet18 é superior no CIFAR-10?

1. **Transfer Learning**: os pesos pré-treinados no ImageNet (~1.2M imagens, 1000 classes) capturam features visuais universais — bordas, texturas, formas — que se transferem bem para o CIFAR-10.
2. **Arquitetura mais profunda**: a ResNet18 tem 18 camadas com *skip connections* (conexões residuais) que permitem treinar redes mais profundas sem degradação de gradientes.
3. **Complexidade do dataset**: imagens coloridas 32x32 com objetos reais (aviões, carros, animais) exigem representações mais ricas do que dígitos manuscritos.

### Confusões Comuns no CIFAR-10

Analisando as matrizes de confusão, é esperado que:
- **cat/dog** sejam confundidos com frequência (formas e texturas semelhantes em baixa resolução)
- **automobile/truck** também apresentem confusão (ambos são veículos)
- **airplane/ship** podem ser confundidos (fundos semelhantes — céu/água)

### Impacto da Regularização

- **Data Augmentation** (RandomCrop + RandomHorizontalFlip): fundamental para o CIFAR-10, previne overfitting e melhora generalização
- **BatchNorm**: estabiliza o treinamento e permite learning rates mais altas
- **Dropout(0.5)**: reduz overfitting no classificador
- **StepLR Scheduler**: reduz o learning rate na metade do treino para refinamento

### Melhorias Possíveis para a CNN Customizada

1. **Arquitetura mais profunda**: adicionar mais blocos convolucionais (4-5 blocos)
2. **Residual connections**: incorporar skip connections inspiradas na ResNet
3. **Data augmentation mais agressiva**: Cutout, MixUp, AutoAugment
4. **Cosine Annealing** como scheduler de learning rate
5. **Mais épocas** (100+), especialmente com augmentation
6. **Redes pré-treinadas mais leves**: EfficientNet-B0 ou MobileNetV3 para um equilíbrio entre desempenho e custo